In [ ]:
# wake word logic

import pyaudio
import numpy as np
from openwakeword.model import Model

def test_open_wake_word():
    # Audio stream parameters
    FORMAT = pyaudio.paInt16
    CHANNELS = 1
    RATE = 16000
    CHUNK = 1280

    audio = pyaudio.PyAudio()
    mic_stream = None
    oww_model = None

    try:
        print("Loading OpenWakeWord models (this takes a few seconds)...")
        
        # BULLETPROOF FIX: Passing no arguments forces it to load the built-in default models
        oww_model = Model()
        
        mic_stream = audio.open(
            format=FORMAT, 
            channels=CHANNELS, 
            rate=RATE, 
            input=True, 
            frames_per_buffer=CHUNK
        )

        print("🎙️ RUA's Ear is Online (100% Local).")
        print("Listening for the wake word: 'Hey Jarvis'...")
        print("(Press Ctrl+C or stop the Jupyter cell to exit)")

        # The Listening Loop
        while True:
            # Read audio from microphone
            audio_chunk = mic_stream.read(CHUNK, exception_on_overflow=False)
            
            # Convert to numpy array for OpenWakeWord
            audio_data = np.frombuffer(audio_chunk, dtype=np.int16)
            
            # Feed audio to the model
            oww_model.predict(audio_data)
            
            # Check the buffer for the Jarvis model
            for mdl in oww_model.prediction_buffer.keys():
                # Filter to only react to the jarvis model, ignore the others (like alexa)
                if "jarvis" in mdl.lower():
                    # If confidence is > 0.5, we have a detection
                    if oww_model.prediction_buffer[mdl][-1] > 0.5:
                        print(f"\n🔔 Wake word '{mdl}' detected! Confidence: {oww_model.prediction_buffer[mdl][-1]:.2f}")
                        # Clear the buffer so it doesn't trigger multiple times
                        oww_model.reset()

    except KeyboardInterrupt:
        print("\nStopping wake word detection.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
    finally:
        if mic_stream is not None:
            mic_stream.close()
        audio.terminate()
        print("Microphone closed.")

# Run the test
test_open_wake_word()

In [ ]:
# memory_management.py
import json
import os
from datetime import datetime
from typing import Dict, Any, List

# Updated file name to reflect single-user architecture
MEMORY_FILE = "rua_memory.json"

class RUACognitiveHub:
    def __init__(self, storage_path: str = MEMORY_FILE):
        self.storage_path = storage_path
        self.memory = self._load_from_disk()
        
        # TIER 1: Working Memory (Session specific)
        self.working_memory = {"buffer": {}, "last_search": None}

    def _load_from_disk(self) -> Dict[str, Any]:
        """Loads memory or initializes a flat, single-user structure."""
        if os.path.exists(self.storage_path):
            with open(self.storage_path, "r") as f:
                return json.load(f)
                
        # Default single-user memory structure
        return {
            "critical_info": {},
            "semantic_memory": {},
            "episodic_memory": [],
            "created_at": datetime.now().strftime("%Y-%m-%d")
        }

    def save_to_disk(self):
        """Saves the current memory state to disk."""
        with open(self.storage_path, "w") as f:
            json.dump(self.memory, f, indent=4)

    def learn_fact(self, key: str, value: Any, tier: str = "semantic"):
        """Stores a fact directly into the single-user memory."""
        if tier == "critical":
            self.memory["critical_info"][key] = value
        else:
            self.memory["semantic_memory"][key] = value
        self.save_to_disk()

    def get_brain_dump(self) -> str:
        """Returns the current cognitive context."""
        dump = "RUA Memory Dump: "
        if self.memory.get('critical_info'): 
            dump += f"CRITICAL: {self.memory['critical_info']}. "
        if self.memory.get('semantic_memory'): 
            dump += f"FACTS: {self.memory['semantic_memory']}. "
        if self.working_memory.get('buffer'): 
            dump += f"ACTIVE TASK: {self.working_memory['buffer']}. "
        return dump

    def consolidate(self, llm):
        """Distills episodic memory into semantic facts."""
        episodes = self.memory.get("episodic_memory", [])
        if len(episodes) < 1: 
            return "Archives lean. No consolidation needed."
        
        prompt = f"Distill these memories: {episodes}. Return JSON facts."
        try:
            res = llm.invoke(prompt)
            # Strip markdown formatting from LLM response
            facts = json.loads(res.content.replace("```json", "").replace("```", "").strip())
            
            self.memory["semantic_memory"].update(facts)
            # Keep only the last episode, clear the rest
            self.memory["episodic_memory"] = self.memory["episodic_memory"][-1:]
            self.save_to_disk()
            return "Brain optimized."
        except Exception as e:
            # Silently fail or log error
            return "Consolidation skipped."

# Instantiate the single-user brain
rua_brain = RUACognitiveHub()

# Note: identity_router has been completely removed as it is no longer needed.

In [ ]:
import speech_recognition as sr
import numpy as np
from faster_whisper import WhisperModel
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.chat_message_histories import ChatMessageHistory
import os, asyncio, edge_tts, pygame, io, threading, queue, time, sys, nest_asyncio

nest_asyncio.apply()
pygame.mixer.init(frequency=24000) # Optimized for Edge TTS sample rate

# --- CONFIGURATION ---
# Use 'tiny.en' on CPU for instant transcription. 
# 'small' is 4x slower than 'tiny'.
STT_MODEL = WhisperModel("tiny.en", device="cpu", compute_type="int8")
voice_queue = queue.Queue()

# --- THE SECRET SAUCE: MEMORY BUFFER AUDIO ---
async def stream_voice_to_mixer(text):
    """Generates audio and plays it from RAM, bypassing the hard drive."""
    voice = "en-IN-NeerjaNeural"
    communicate = edge_tts.Communicate(text, voice, rate="+15%") # Fast & Witty
    
    # We use a BytesIO buffer to avoid the slow 'save to disk' step
    audio_data = b""
    async for chunk in communicate.stream():
        if chunk["type"] == "audio":
            audio_data += chunk["data"]
    
    if audio_data:
        # Load audio directly from memory
        sound_file = io.BytesIO(audio_data)
        pygame.mixer.music.load(sound_file)
        
        # Calculate sync-print timing
        words = text.split()
        # Assume average speech rate of 3 words per second for typing effect
        # This makes the text appear as the voice speaks
        pygame.mixer.music.play()
        
        for word in words:
            sys.stdout.write(word + " ")
            sys.stdout.flush()
            time.sleep(0.18) # Matches +15% speech rate roughly
            
        while pygame.mixer.music.get_busy():
            pygame.time.Clock().tick(60)

def voice_worker():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    while True:
        text = voice_queue.get()
        if text is None: break
        loop.run_until_complete(stream_voice_to_mixer(text))
        voice_queue.task_done()

threading.Thread(target=voice_worker, daemon=True).start()

def start_rua_hybrid_master():
    # Use Gemini for cloud tasks (it has faster 'Time to First Token' than Llama local)
    cloud_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)
    local_llm = ChatOllama(model="llama3.2", temperature=0.1)
    session_history = ChatMessageHistory()
    recognizer = sr.Recognizer()
    
    # Optimization: Faster silence detection
    recognizer.pause_threshold = 0.5  # Stops listening 0.5s after you stop talking
    recognizer.non_speaking_duration = 0.3

    print("🚀 RUA TURBO: ZERO-LATENCY MODE ON")

    try:
        with sr.Microphone(sample_rate=16000) as source:
            # Calibrate once and keep it
            recognizer.adjust_for_ambient_noise(source, duration=0.5)
            
            while True:
                print("\n👂 Listening...", end="", flush=True)
                audio = recognizer.listen(source, timeout=None, phrase_time_limit=8)
                
                # --- Instant STT ---
                audio_np = np.frombuffer(audio.get_raw_data(), dtype=np.int16).astype(np.float32) / 32768.0
                segments, _ = STT_MODEL.transcribe(audio_np, beam_size=1)
                full_user_text = "".join([s.text for s in segments]).strip()
                
                if full_user_text:
                    print(f"\r👤: {full_user_text}")
                    print(f"🤖 RUA: ", end="", flush=True)
                    
                    messages = [
                        {"role": "system", "content": "You are RUA. Witty, fast, brief. No markdown. < 20 words."},
                        {"role": "user", "content": full_user_text}
                    ]

                    # Use Gemini-Flash for speed unless it's a simple local query
                    llm = cloud_llm if len(full_user_text) > 10 else local_llm
                    
                    sentence_buffer = ""
                    for chunk in llm.stream(messages):
                        content = chunk.content if hasattr(chunk, 'content') else str(chunk)
                        sentence_buffer += content

                        # Aggressive trigger: send to voice at the first sign of a sentence
                        if any(p in sentence_buffer for p in [".", "!", "?", "\n"]):
                            voice_queue.put(sentence_buffer.strip())
                            sentence_buffer = ""

                    if sentence_buffer.strip():
                        voice_queue.put(sentence_buffer.strip())
                else:
                    print("\r", end="")

    except KeyboardInterrupt:
        print("\n🛑 Off.")

if __name__ == "__main__":
    start_rua_hybrid_master()